# 05 · End-to-end LLM (paper §3)

The paper's §3 proposes a language model **without a task head** that reads the patient
record and answers with one token per time unit — `[N]` no event, `[C]` censored,
`[Ek]` event k (and combination tokens like `[E1+E3]` when several events share a
unit) — trained with token-level cross-entropy (Eq. 8). Implemented in `sat/llm.py`:

* prompt: the record as text (`x_age 0.53, c_sex_F, ...`, or the coded history);
* answer units = the same duration cuts our PC-hazard model uses;
* per-event curves: teacher-force an all-`[N]` answer, read the next-token
  probabilities at each unit, remove the censoring mass → event hazards → S_k, CIF_k;
* scored with the same C_td / Brier / within-subject / MAE metrics.

Two flavours: **pretrained DistilGPT2** (the paper's "any pretrained LLM") and a
**small GPT-2 trained from scratch** with a BPE vocabulary fitted on the records.
References on the same splits: Cox, our transformer (L_PCH and the paper's recipe),
oracle (simulator). Needs a GPU session and Internet (model download).

## Options

In [ ]:
NAME = "05_llm_end_to_end"

# Datasets: two multi-event (simulator with oracle, real EBMT) + METABRIC (single event,
# comparable with notebook 03 and SurvTRACE).
DATASETS = ["sim_base", "ebmt", "metabric"]
SEEDS    = [0, 1, 2]     # seed s = init seed s AND split s (same splits as notebooks 01/03)

# The paper's §3 model in two flavours:
LLMS = {
    "llm_pretrained": ["llm_init=pretrained", "llm_model=distilgpt2", "llm_learning_rate=5e-5"],
    "llm_scratch":    ["llm_init=scratch", "llm_learning_rate=5e-4"],
}
LLM_EPOCHS = 30          # early stopping on the validation answer cross-entropy (patience 3)

# References on the same splits, so the table is self-contained:
REFERENCE_BASELINES = ["coxph"]
REFERENCE_RECIPES   = ["nllpch", "nllpch_sample_event_ranking"]   # our transformer

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/new/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Datasets

In [ ]:
# ---- datasets -------------------------------------------------------------------------
def ensure_dataset(ds):
    """Simulated scenarios and EBMT are generated on the fly; hsa_synthetic ships with the repo."""
    if ds == "hsa_synthetic" or (REPO / "data" / ds / "meta.json").is_file():
        return
    if ds == "ebmt":
        subprocess.run([sys.executable, "scripts/prepare_ebmt.py"], check=True)
    elif ds.startswith("sim_"):
        subprocess.run([sys.executable, "-m", "sat.data.simulate", "--scenario", ds[4:],
                        "--out", f"data/{ds}"], check=True, stdout=subprocess.DEVNULL)
    else:
        raise ValueError(ds)

def group(ds):
    return "hsa_synthetic" if ds == "hsa_synthetic" else "multievent"

def ds_overrides(ds):
    # hsa keeps its own configs; score every event at its own horizons like the others
    return ["per_event_horizons=true"] if ds == "hsa_synthetic" else dataset_overrides(ds, REPO)

def has_truth(ds):
    return (REPO / "data" / ds / "truth.npz").is_file()

SINGLE = {"metabric": ("survtrace_metabric", "metabric_numeric")}

def group5(ds):
    return SINGLE[ds][0] if ds in SINGLE else group(ds)

def hub5(ds):
    return SINGLE[ds][1] if ds in SINGLE else ds

def ov5(ds):
    return [] if ds in SINGLE else ds_overrides(ds)

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
for ds in DATASETS:
    if ds not in SINGLE:
        ensure_dataset(ds)
    r.prepare(f"{group5(ds)}/survival", ov5(ds), label=ds)

## Plan and run

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for ds in DATASETS:
        g, hub, dov, base = group5(ds), hub5(ds), ov5(ds), {"dataset": ds, "seed": s}
        for name, llm_ov in LLMS.items():
            runs.append(Run(f"{ds}__{name}__s{s}", "llm", f"{g}/survival",
                            dov + so + llm_ov + [f"llm_epochs={LLM_EPOCHS}"], hub, base | {"model": name}))
        for m in REFERENCE_BASELINES:
            runs.append(Run(f"{ds}__{m}__s{s}", "coxph" if m == "coxph" else "finetune",
                            f"{g}/{m}", dov + so, hub, base | {"model": m}))
        for rec in REFERENCE_RECIPES:
            runs.append(Run(f"{ds}__ours_{rec}__s{s}", "finetune", f"{g}/survival",
                            dov + so + [f"tasks/losses={rec}"], hub, base | {"model": "ours", "recipe": rec}))
        if has_truth(ds):
            runs.append(Run(f"{ds}__oracle__s{s}", "oracle", f"{g}/survival", dov + so, hub, base | {"model": "oracle"}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
The LLM rows are scored with exactly the metrics, splits and horizons of the other rows, from per-event curves derived from its answer-token probabilities (see `sat/llm.py`).

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)